In [ ]:
'''
TRAINING DATASET
Narrow Band Imaging (NBI) images of each detected lesion
non-magnified NBI images, originating from 1,339 unique polyps (73% adenomas, 17% hyperplastic polyps, 10% sessile serrated lesions) detected during 555 different colonoscopies.

VALIDATION DATASET
For each polyp, location, size and morphology (Paris classification) and predicted histology (optical diagnosis) were recorded.

'''

In [ ]:
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import filedialog
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import os
from common import get_image_size, move_images
import shutil

In [ ]:
# Open file dialog to select the spreadsheet file
spreadsheet_path = filedialog.askopenfilename(
    title="Select the Spreadsheet File",
    filetypes=[("Excel files", "*.xlsx")]
)

# Only annotation on validation data includes information on polyp morphology
validation_folder_path = filedialog.askdirectory(title="Select Validation Set Folder")

In [ ]:
# Read the spreadsheet
spreadsheet = pd.ExcelFile(spreadsheet_path)
# Read validation data 
polar_data = pd.read_excel(spreadsheet, 'validation_set')

In [ ]:
polar_data[['WIDTH', 'HEIGHT']] = None

for idx, row in polar_data.iterrows():
        patient_folder = os.path.join(validation_folder_path, row['PATIENT_ID'])
        image_id = row['POLYP/IMAGE_ID']
        image_p = os.path.join(patient_folder, image_id + ".png")
        if os.path.exists(image_p):
            properties = get_image_size(image_p)
            polar_data.loc[idx, ['WIDTH', 'HEIGHT']] = properties
polar_data.head()

In [ ]:
polar_data.shape[0]

In [ ]:
# Number of unique polyps and patients

In [ ]:
unique_polyps_validation = polar_data['POLYP/IMAGE_ID'].nunique()
unique_patients_validation = polar_data['PATIENT_ID'].nunique()

unique_polyps_validation, unique_patients_validation

In [ ]:
# Number of polyps per image

In [ ]:
# Count the number of polyps per image in the validation set, using the correct column name for polyp identification
polyps_per_image_validation = polar_data.groupby('POLYP/IMAGE_ID')['POLYP/IMAGE_ID'].nunique()

# Calculate the average number of polyps
avg_polyps_per_image_validation = polyps_per_image_validation.mean()

avg_polyps_per_image_validation, polyps_per_image_validation.describe()


In [ ]:
# Distribution of polyp sizes. Figures

In [ ]:
# Calculate the distribution of polyps by size for the validation set
size_distribution_validation = polar_data['POLYP SIZE'].value_counts()

size_distribution_validation

In [ ]:
# Distribution of polyp sizes. Histogram (negative values excluded)

In [ ]:
# Remove entries with negative values

polar_data = polar_data[polar_data['POLYP SIZE'] >= 0]
size_distribution_filtered = polar_data['POLYP SIZE'].value_counts().sort_index()

# Plot histograms for the distribution of polyps by size for both sets
plt.figure(figsize=(15, 6))
size_distribution_filtered.plot(kind='bar', color='orange')
plt.title('Distribution of Polyps by Size')
plt.xlabel('Polyp Size')
plt.ylabel('Frequency')
plt.grid(axis='y')

plt.tight_layout()
plt.show()

In [ ]:
polar_data.shape[0]

In [ ]:
# Distribution of polyp sizes. Descriptive statistics (excluding negative values)

In [ ]:
polar_data['POLYP SIZE'].describe()

In [ ]:
# Generate vertical box plots for the polyp sizes in the training and validation sets
plt.figure(figsize=(12, 6))
polar_data['POLYP SIZE'].plot(kind='box', color='orange')
plt.title('Box Plot of Polyp Sizes')
plt.ylabel('Size')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate the distribution of polyps by pathology diagnosis
pathology_distribution = polar_data['PATHOLOGY DIAGNOSIS'].value_counts()

# Generate histograms for the distribution of polyps by pathology diagnosis
plt.figure(figsize=(15, 6))
pathology_distribution.plot(kind='bar', color='orange')
plt.title('Pathology Diagnosis Distribution - Validation Set')
plt.xlabel('Pathology Diagnosis')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of polyps per patient.

In [ ]:
polyps_per_patient_validation_original = polar_data.groupby('PATIENT_ID')['POLYP/IMAGE_ID'].count()
polyps_per_patient_validation_distribution_original = polyps_per_patient_validation_original.value_counts().sort_index()

# Calculate the descriptive statistics for the number of polyps per patient in the original sets
desc_stats_polyps_per_patient_validation_original = polyps_per_patient_validation_original.describe()

# Generate histograms for the revised distribution of polyps per patient
plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 2)
polyps_per_patient_validation_distribution_original.plot(kind='bar', color='orange')
plt.title('Polyps per Patient Distribution')
plt.xlabel('Number of Polyps')
plt.ylabel('Number of Patients')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Location analysis. Descriptive statistics

In [ ]:
# Function to parse the polyp position and extract x and y coordinates
def extract_coordinates(positions):
    coordinates = positions.str.strip('[]').str.split(',', expand=True).astype(int)
    x = coordinates[0]
    y = coordinates[1]
    return x, y

# Extract coordinates
x_original, y_original = extract_coordinates(polar_data['POLYP POSITION (x,y,w,h)'])

# Generate descriptive statistics for the polyp positions
desc_stats_positions_original = pd.concat([x_original, y_original], axis=1).describe()

desc_stats_positions_original

In [ ]:
# We identify the entries with negative coordinate values

In [ ]:
# Function to check for negative values in the position data
def has_negative_values(position):
    coordinates = position.strip('[]').split(',')
    return any(int(coord) < 0 for coord in coordinates)

# Identifying the entries with negative coordinates
negative_coords = polar_data[polar_data['POLYP POSITION (x,y,w,h)'].apply(has_negative_values)]

# Displaying the entries with negative coordinates along with Patient ID
negative_coords[['PATIENT_ID', 'POLYP/IMAGE_ID', 'POLYP POSITION (x,y,w,h)']]

In [ ]:
# Location analysis. Are there bounding boxes that exceed the boundaries of the images? 

In [ ]:
def check_boundary_overflow(df):
    """
    Function to check if the polyp locations exceed the image boundaries.
    """
    # Extracting polyp positions and converting them to numeric values
    df['x'], df['y'], df['w'], df['h'] = zip(*df['POLYP POSITION (x,y,w,h)'].apply(lambda x: eval(x)))

    # Checking if the polyp locations exceed the image boundaries
    df['OVERFLOWS_WIDTH'] = (df['x'] + df['w']) > df['WIDTH']
    df['OVERFLOWS_HEIGHT'] = (df['y'] + df['h']) > df['HEIGHT']

    # Returning rows where either width or height is exceeded
    return df[(df['OVERFLOWS_WIDTH']) | (df['OVERFLOWS_HEIGHT'])]

def update_boundary_overflow_reason(df):
    """
    Update the boundary overflow reason to handle cases where both width and height are exceeded.
    """
    def determine_overflow_reason(row):
        if row['OVERFLOWS_WIDTH'] and row['OVERFLOWS_WIDTH']:
            return 'W+H'
        elif row['OVERFLOWS_WIDTH']:
            return 'W'
        elif row['OVERFLOWS_WIDTH']:
            return 'H'
        else:
            return 'None'

    df['OVERFLOW'] = df.apply(determine_overflow_reason, axis=1)
    return df

# Checking for boundary overflow
polar_data_w_overflow = check_boundary_overflow(polar_data.copy())
polar_data_w_overflow = update_boundary_overflow_reason(polar_data_w_overflow)

polar_data_w_overflow


In [ ]:
# We need to calculate the maximum value for these coordinates to fix this issue

In [ ]:
def adjust_polyp_dimensions_override(df):
    """
    Adjust the width or height of the polyp position if they exceed the image boundaries and override the original values.
    """
    # Extracting polyp positions and converting them to numeric values
    df['x'], df['y'], df['w'], df['h'] = zip(*df['POLYP POSITION (x,y,w,h)'].apply(lambda x: eval(x)))

    # Adjusting width and height if they exceed image boundaries
    df['w'] = df.apply(lambda row: min(row['w'], row['WIDTH'] - row['x']), axis=1)
    df['h'] = df.apply(lambda row: min(row['h'], row['HEIGHT'] - row['y']), axis=1)

    # Updating the POLYP POSITION column with the adjusted values
    df['POLYP POSITION (x,y,w,h)'] = df.apply(lambda row: f"[{row['x']}, {row['y']}, {row['w']}, {row['h']}]", axis=1)

    return df

polar_data = adjust_polyp_dimensions_override(polar_data)

In [ ]:
# Checking if there is still boundary overflow
polar_data_w_overflow = check_boundary_overflow(polar_data.copy())
polar_data_w_overflow = update_boundary_overflow_reason(polar_data_w_overflow)

polar_data_w_overflow

In [ ]:
# Polyp morphology

In [ ]:
# Analyzing the distribution of POLYP MORPHOLOGY
polar_data['POLYP MORPHOLOGY'].value_counts()

In [ ]:
polar_data = polar_data[~polar_data['POLYP MORPHOLOGY'].isin(['##USER_MISSING_97##', '##USER_MISSING_99##', 'Stenosing'])]

In [ ]:
morphology_to_code = {
    'Sessile': 'Is',
    'Flat elevated': 'IIa',
    'Pedunculated': 'Ip',
    'Flat': 'IIb',
    'Depressed': 'IIc'
}

# Replace the 'POLYP MORPHOLOGY' values based on the mapping
polar_data['POLYP MORPHOLOGY'] = polar_data['POLYP MORPHOLOGY'].replace(morphology_to_code)
polar_data.rename(columns={'POLYP MORPHOLOGY': 'SHAPE'}, inplace=True)

In [ ]:
# Copy images from the subdirectories to a target img folder

In [ ]:
# destination path
destination_image_directory = os.path.dirname(spreadsheet_path) + '/img_temp/'
# Create the base destination folder if it doesn't exist
os.makedirs(destination_image_directory, exist_ok=True)

for idx, row in polar_data.iterrows():
        patient_folder = os.path.join(validation_folder_path, row['PATIENT_ID'])
        image_id = row['POLYP/IMAGE_ID']
        source_file = os.path.join(patient_folder, image_id + ".png")
        destination_file = os.path.join(destination_image_directory, image_id + ".png")
        if os.path.exists(source_file):
            shutil.copy(source_file, destination_file)

In [ ]:
polar_data.drop(columns=['PATIENT_ID', 'POLYP LOCATION', 'OPTICAL DIAGNOSIS', 'OPTCAL DIAGNOSIS CONFIDENCE', 'PATHOLOGY DIAGNOSIS', 'x', 'y', 'w', 'h'], inplace=True)

In [ ]:
# Frame field

In [ ]:
polar_data.rename(columns={'POLYP/IMAGE_ID': 'FRAME'}, inplace=True)
polar_data['FRAME'] = polar_data['FRAME'] + '.png'

In [ ]:
# Size field

In [ ]:
polar_data.rename(columns={'POLYP SIZE': 'SIZE'}, inplace=True)

In [ ]:
# Image type

In [ ]:
polar_data['IMAGE TYPE'] = 'NBI'

In [ ]:
# Bounding boxes

In [ ]:
# Transform the 'BOUNDING BOX' values
def transform_bounding_box(bbox_str):
    # Strip the square brackets, split by commas, and convert to integers
    bbox_components = list(map(int, bbox_str.strip('[]').split(',')))
    x, y, width, height = bbox_components
    # Calculate x max and y max
    x_max = x + width
    y_max = y + height
    # Return the formatted string
    return f"{x},{y},{x_max},{y_max}"

polar_data.rename(columns={'POLYP POSITION (x,y,w,h)': 'BOUNDING BOX'}, inplace=True)
polar_data['BOUNDING BOX'] = polar_data['BOUNDING BOX'].apply(transform_bounding_box)

In [ ]:
polar_data.head()

In [ ]:
# Saving the data to a spreadsheet
save_path = os.path.join(os.path.dirname(spreadsheet_path), 'annotations.xlsx')
polar_data.to_excel(save_path, index=False)

In [ ]:
len(polar_data)

In [ ]:
source_image_directory = os.path.dirname(spreadsheet_path) + '/img_temp/'
destination_image_directory = os.path.dirname(spreadsheet_path) + '/img/'
move_images(source_image_directory, destination_image_directory, polar_data)